# Base 5 — Ouro

## Documentação técnica e interpretativa da V1

Este capítulo trata **exclusivamente** da base Ouro. Reúne as evidências congeladas da V1 para posterior incorporação ao relatório das cinco bases, sem comparar bases, calcular vitórias globais ou alterar resultados oficiais. As células leem apenas os artefatos existentes em `analyses/grupo5/outputs/`; não executam tuning nem walk-forward.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

def find_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'analyses' / 'grupo5' / 'outputs').exists():
            return candidate
    raise FileNotFoundError('Execute o notebook dentro do repositório pls-regration.')

ROOT = find_root()
OUT = ROOT / 'analyses' / 'grupo5' / 'outputs'

def table(name, columns=None):
    df = pd.read_csv(OUT / name)
    return df if columns is None else df.loc[:, columns]

def figure(relative_path, width=1000):
    path = OUT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    display(Image(filename=str(path), width=width))

print('Artefatos congelados:', OUT.relative_to(ROOT))

## 1. Visão geral da base

A tarefa prevê `TARGET`, o preço do ouro na próxima observação real do calendário bruto, a partir da origem atual. A série representa o fixing da tarde de Londres da Deutsche Bundesbank (`BBEX3.D.XAU.USD.EA.AC.C05`), em **USD por onça fina (troy ounce)**. Ouro é um ativo financeiro e reserva de valor; a evidência V1 permite observar sua dinâmica histórica, não provar causalidade econômica.

A fonte cobre 01/04/1968 a 10/04/2014, em dias de mercado. As externas foram Treasury de 10 anos (`DGS10`, FRED) e Federal Funds Rate (`DFF`, FRED), disponibilizadas de forma conservadora. A base modelável final tem 9.864 observações e 12 features causais.

## 2. Documentação e qualidade dos dados

Foram lidos 12.009 registros brutos; 11.641 têm preço válido. Há 368 preços ausentes, nenhuma data duplicada e 9.864 linhas após exigir alvo, lags, janelas e externas disponíveis. O ouro **não foi interpolado**: a lacuna permanece ausente e a linha não entra na base modelável.

O calendário é irregular: há 401 intervalos acima de três dias e o maior intervalo é de 20 dias. Extremos em nível (IQR) e retorno (MAD) foram preservados por não haver evidência de erro. As externas foram alinhadas por `left merge` no calendário do ouro, receberam somente `forward fill` e, então, `shift(1)`. Não há backfill.

## 3. Análise exploratória

A série tem mudanças persistentes de nível e inclinação, portanto não representa crescimento constante. A distribuição e os movimentos extremos apontam para regimes de mercado. Treasury 10Y e Fed Funds exibem seus próprios patamares; a comparação normalizada é descritiva e não demonstra causalidade. A mediana entre preços válidos é um dia, e o término em 2014 limita qualquer generalização a períodos posteriores.

In [ ]:
display(table('exploratory_summary.csv'))
display(table('data_quality_audit.csv'))
figure('graficos/gold_price_full.png')
figure('01_data_quality_outliers.png')
figure('graficos/treasury_10y_full.png')
figure('graficos/fed_funds_rate_full.png')
figure('graficos/series_normalized_comparison.png')
figure('03_distribuicoes_treino.png')

## 4. STL e sazonalidade

A STL foi estimada somente no treino e separa observado, tendência, sazonalidade e resíduo. Foram testados `m=5`, `m=20` e `m=252`, hipóteses de semana de pregões, aproximadamente mês e aproximadamente ano. A tendência é forte nos três ajustes; a força sazonal foi **0,0** em todos eles.

Não há evidência de sazonalidade forte. `m=5` permaneceu como referência operacional de semana de mercado em alguns experimentos, não como uma sazonalidade comprovada. Períodos maiores não tornaram a decomposição sazonal convincente e elevaram a dispersão residual.

## 5. Feature Engineering

RF e PLS usam as mesmas 12 features causais. `GOLD_PRICE` é o nível observado na origem; lags trazem histórico; média e desvio móveis resumem os cinco preços anteriores; seno/cosseno codificam calendário. Treasury 10Y e Fed Funds compõem o contexto econômico disponível antes da origem. As janelas usam `GOLD_PRICE.shift(1)` e não incluem o preço da origem. `TARGET` é o alvo e `TARGET_UP` não entra na base modelada.

## 6. Prevenção de leakage

Uma origem em `t` prevê `TARGET`, o próximo preço do calendário bruto. A auditoria reconstruiu as 9.864 linhas: `TARGET = GOLD_PRICE.shift(-1)` é criado antes do filtro final. Por isso, em 428 casos o alvo não é a próxima linha modelável, mas continua sendo o próximo preço do calendário bruto.

Não há backfill; externas recebem preenchimento para frente e defasagem; rolling features são causais; `TARGET` e `TARGET_UP` não são features. O PLS ajusta um novo `StandardScaler` apenas no treino de cada janela. A auditoria não encontrou leakage, desalinhamento ou previsões não finitas.

In [ ]:
display(table('stl_candidates.csv'))
figure('stl_period_5.png')
display(table('feature_dictionary.csv', ['nome','tipo','calculo','informacao_usada','conhecida_na_origem']))
display(table('leakage_feature_checks.csv'))
display(table('prediction_file_checks.csv'))

## 7. Protocolo experimental

A base modelável possui 9.864 observações: treino cronológico de 70% (6.904), validação de 15% (1.480) e teste de 15% (1.480). O walk-forward usa janela expansiva, horizonte de uma observação futura e passo cinco. Foram congeladas 296 origens de validação e 296 de teste.

Os quatro modelos usam as mesmas origens, datas-alvo e `Y_TRUE`, tornando o MAE comparável dentro da base Ouro. Hiperparâmetros foram escolhidos apenas na validação; o teste final não reabriu nenhuma decisão.

## 8. Modelos avaliados

**SARIMAX** combina termos ARIMA, componente sazonal curto e externas/calendário. **Holt-Winters** é a referência univariada de nível, tendência e sazonalidade. **Random Forest** captura não linearidades nas 12 features. **PLS Regression** projeta as 12 features em componentes latentes orientados pela covariância com o alvo. As configurações abaixo venceram na validação e foram congeladas antes do teste.

## 9. Otimização de hiperparâmetros

SARIMAX fez 72 ajustes de treino: 18 ordens `(p,d,q)` e quatro estruturas sazonais de período 5. BIC selecionou os cinco para triagem, mas o vencedor foi definido pelo MAE walk-forward: `(0,1,2) × (0,0,1,5)`. Holt-Winters comparou 15 estruturas e venceu com tendência/sazonalidade aditivas, período 5 e sem damping; a suavização é estimada em cada janela.

RF comparou 12 configurações dirigidas com `random_state=42`; venceu 300 árvores, profundidade 20, `min_samples_split=10`, `min_samples_leaf=5`, `max_features=1.0`. PLS testou 1–12 componentes e selecionou 11. Nenhuma decisão usou o teste final.

In [ ]:
figure('graficos/walkforward_split_timeline.png')
display(table('hyperparameters.csv', ['model','parameters','validation_mae','n_candidates','selection_criterion']))
display(table('sarimax_bic_screening.csv').sort_values('bic').head(5))
display(table('tuning_candidates_Holt_Winters.csv').dropna(subset=['validation_mae']).head(5))
display(table('tuning_candidates_Random_Forest.csv').sort_values('validation_mae').head(5))
figure('pls_validation_components.png')

## 10. Resultados por MAE — Ouro

MAE é a métrica oficial e está em USD por onça troy. No teste final, Holt-Winters teve o menor MAE (11,5331), seguido de SARIMAX (11,5988), PLS (11,6002) e Random Forest (14,2826). Este ranking vale **somente para Ouro**.

## 11. Contextualização do erro

Nas mesmas origens de teste, o movimento absoluto médio entre origem e alvo foi **US$ 11,6377/oz** e a mediana foi **US$ 8,50/oz**. Persistência teve MAE 11,6376689189. Para Holt-Winters, `11,5331 / 11,6377 ≈ 0,991`: o MAE do melhor equivale a aproximadamente 99,1% da variação absoluta média observada. Holt-Winters é formalmente o menor MAE, mas a melhora sobre persistência é inferior a 1%; não há evidência de vantagem preditiva muito grande.

## 12. Baseline de persistência

Persistência é diagnóstico adicional, não quinto modelo oficial: `P(t+1) = P(t)`. Ela é importante para preços financeiros e usa as mesmas 296 origens, datas-alvo e observações reais. O diagnóstico consecutivo de 1.480 linhas usa outra amostra e não entra na comparação oficial.

In [ ]:
display(table('metrics.csv').sort_values('mae')[['model','n_forecasts','mae','rank_gold']])
display(table('model_comparison_vs_baseline.csv').sort_values('mae'))
figure('mae_comparison_gold.png')

## 13. Resíduos

Resíduos fora da amostra são `real − previsto`. Holt-Winters e PLS tiveram média próxima de zero e 0 rejeições em 20 lags de Ljung-Box: os resíduos são compatíveis com ausência de autocorrelação significativa nos lags avaliados. SARIMAX teve uma rejeição isolada; é evidência pontual. RF teve três rejeições, maior dispersão e viés médio positivo, sinalizando estrutura residual remanescente e subestimação média. A tabela completa de Ljung-Box permanece no corpo deste notebook para futura consolidação.

## 14. Importância das features

No RF, a importância nativa foi dominada por `GOLD_PRICE` (0,9013) e `GOLD_LAG_1` (0,0926); a permutação confirma ambos no topo. No PLS, coeficientes padronizados, VIP e permutação destacam `GOLD_PRICE`, `GOLD_LAG_1`, `GOLD_ROLLING_MEAN_5`, `GOLD_LAG_5` e `GOLD_LAG_20`. Importância não é causalidade. Treasury/Fed Funds terem menor contribuição marginal nesta modelagem não significa que juros não influenciem ouro em outros horizontes ou regimes.

In [ ]:
display(table('residual_diagnostics_summary.csv'))
display(table('ljung_box.csv'))
figure('residual_holt_winters_serie.png')
figure('residual_holt_winters_acf.png')
figure('ljung_box_comparison.png')
importance = table('feature_importance_summary.csv')
display(importance.sort_values(['model', 'permutation_rank']).groupby('model', group_keys=False).head(5))
figure('feature_importance_random_forest.png')
figure('feature_importance_pls_regression.png')

## 15. Estudo aprofundado do PLS

Partial Least Squares (PLS) constrói componentes latentes: combinações lineares de `X` que resumem as features e sua covariância com `y`. Isso é útil quando preço atual, lags e médias móveis são correlacionados. Em vez de estimar diretamente no espaço original, PLS usa direções orientadas pelo alvo; `n_components` regula quanto dessa estrutura é retida.

O PLS usou as mesmas 12 features causais do RF e padronização ajustada novamente em cada treino walk-forward. `TARGET` e `TARGET_UP` ficaram fora das features. O artefato `tuning_candidates_PLS_Regression.csv` registra MAE de validação **2,9701** (2,9701247444) para 11 componentes: são 296 origens de validação, com horizonte 1, entre 27/06/2000 e 21/05/2007, na mesma unidade do alvo (USD/oz). O MAE final de teste **11,6002** (11,6002116901) vem de outras 296 origens, de 06/06/2007 a 03/04/2014, em `metrics.csv`. Portanto, os valores não são escalas diferentes nem devem ser comparados como melhoria ou piora direta: pertencem a blocos temporais distintos, com variação de preços distinta; a validação escolheu os componentes e o teste apenas avaliou a configuração congelada. Para 12 features, a compressão é pequena — não automaticamente um problema, mas um sinal de que a informação validada ficou distribuída por várias direções latentes.

Coeficientes padronizados comparam magnitudes, mas seus sinais exigem cautela sob colinearidade. VIP mede contribuição aos componentes que explicam o alvo; permutation importance mede a piora de MAE ao embaralhar uma feature no modelo fixo. Vantagens: lidar com colinearidade e manter uma forma linear relativamente interpretável. Limitações: dependência de escala, relações lineares, componentes menos diretos e possível instabilidade entre regimes. No teste, PLS teve MAE 11,6002, praticamente empatado com SARIMAX e 0,0671 acima de Holt-Winters.

## 16. Limitações da abordagem V1

Persistência é forte e o ganho do melhor modelo é pequeno. Há forte dependência de preço recente, horizonte de uma observação, calendário irregular e histórico que termina em 2014. A referência histórica completa teve movimento diário absoluto médio de aproximadamente 3,8893 USD/oz; no teste oficial de 296 origens, esse valor foi 11,6377 USD/oz e a mediana absoluta foi 8,50 USD/oz. A diferença entre o MAE de validação do PLS (2,9701) e seu MAE de teste (11,6002) não representa mudança de unidade: são períodos temporais com escala e volatilidade muito distintas. Isso é evidência de mudança relevante na dificuldade da série, não prova causal de mudança de regime.

As externas usam disponibilidade conservadora, não vintages intradiários; importância não implica causalidade; e relações podem mudar por regime. A estabilidade temporal das importâncias não foi adequadamente avaliada em múltiplos períodos independentes. A V2 também identificou que os outputs brutos rotularam `target_date` com a data da origem: a consolidação reconciliou a data-alvo pelo manifesto V1 congelado, sem alterar previsões, `Y_TRUE`, as 296 origens ou os MAEs. Esses limites delimitam a interpretação, não invalidam os resultados V1.

## 17. Investigação complementar V2

A V2 foi concluída como estudo complementar, sem reabrir o ranking oficial V1. Ela investigou dificuldade do target, LEVEL/DELTA/LOG_RETURN, ablation, dependência de `GOLD_PRICE`, dólar, juros, risco, commodities, drift, redundância e extrapolação do Random Forest. O PLS/LEVEL/A2 reproduziu a V1: MAE de 11,6002 USD/oz, 0,0671 USD/oz (cerca de 0,58%) acima de Holt-Winters V1 e 0,0375 USD/oz (cerca de 0,32%) melhor que persistência. Os três primeiros modelos V1, portanto, permanecem muito próximos.

O diagnóstico RF em LEVEL encontrou `y_train` entre 252,90 e 725,75 USD/oz e `y_test` entre 647,75 e 1.877,75 USD/oz; em 281 de 296 origens (94,9%), o alvo real superou o máximo visto no treino. Random Forest não extrapola naturalmente níveis além dos valores observados nas folhas. Isso ajuda a explicar seu mau desempenho em LEVEL, mas não é a única causa: também foram observados drift, redundância e mudanças nas distribuições de NASDAQCOM, WTI, DTWEXB, VIX e volatilidades do ouro. Exemplos de redundância incluem DGS2 × `dgs2_lag1` (correlação aproximada de 0,9997), VIXCLS × `vix_lag1` (0,9818) e `distance_ma20` × `gold_momentum_10` (0,9289).

| RF A8 — target | MAE reconstruído (USD/oz) |
|---|---:|
| LEVEL | 39,5007 |
| DELTA | 12,1579 |
| LOG_RETURN | 11,7850 |

A transformação do alvo reduziu drasticamente a dificuldade do RF A8 em relação ao nível crescente; DELTA e LOG_RETURN, porém, não superaram persistência nessa configuração. O melhor resultado experimental foi RF/LOG_RETURN/A3 (`gold_technical_no_raw`), com MAE reconstruído de 11,2986 USD/oz, ganho de 0,3390 USD/oz (cerca de 2,91%) sobre persistência e directional accuracy de cerca de 53,72%. Ele é classificado como **INFORMATIVE**, pois surgiu em especificação experimental posterior e não substitui o ranking V1. A maior directional accuracy observada foi RF/DELTA/A3, 56,42%, métrica complementar — MAE continua sendo a medida principal.

No RF LEVEL FULL, remover `GOLD_PRICE` elevou o MAE em 2,0914 USD/oz (cerca de 14,0%); em horizonte de uma observação, essa dependência autorregressiva é esperada, não um simples “vício”. Remover os lags do ouro trouxe melhora pequena e ainda inferior à persistência, insuficiente para justificar nova seleção. Exógenas sozinhas tiveram MAE de 38,1406 USD/oz: nesta base, período, horizonte e configuração, não substituíram a dinâmica do ouro. No caminho incremental, USD foi o maior auxílio observado, juros acrescentaram informação em algumas configurações e risco prejudicou o caminho analisado; isso não é uma generalização econômica universal. PLS A2 usou 11 componentes para 12 features (91,7%), sem redução dimensional relevante. A estabilidade temporal das importâncias permanece **NOT_ASSESSABLE_SINGLE_TEST_PERIOD**: drift e permutation importance foram apenas descritivos.

## 18. Conclusão da base Ouro

A modelagem oficial principal permanece a V1: (1) Holt-Winters, 11,5331 USD/oz; (2) SARIMAX, 11,5988; (3) PLS, 11,6002; (4) Random Forest, 14,2826. Persistência marcou 11,6377 USD/oz. Holt-Winters, SARIMAX e PLS continuam praticamente empatados, enquanto RF em LEVEL permanece limitado pela extrapolação, maior dispersão e estrutura residual remanescente.

A V2 não substitui esse ranking nem adiciona um quinto modelo oficial. Ela explica limitações do RF em LEVEL, mostra que targets DELTA e LOG_RETURN são uma evolução promissora para investigação futura, evidencia a dependência esperada de `GOLD_PRICE`, e mostra que mais features e novas exógenas não implicaram melhora consistente. RF/LOG_RETURN/A3 é evidência complementar promissora, não sucessor de Holt-Winters. Assim, V1 permanece a referência principal e V2 é uma proposta de evolução metodológica a ser avaliada em protocolo futuro, com períodos independentes, dados mais recentes e seleção pré-teste.

## 19. Referências da base Ouro

- **Fonte institucional e metadados da série:** Deutsche Bundesbank, série `BBEX3.D.XAU.USD.EA.AC.C05`, ouro no fixing da tarde em Londres. O identificador, a descrição e a unidade estão registrados em `outputs/protocol.json`.
- **Aquisição do arquivo usado na V1:** `dengyishuo/quantitative-finance`, arquivo `gold.daily.prices.csv`, no URL de aquisição registrado em `outputs/protocol.json`. O GitHub é o local de download do CSV; não é apresentado como fonte institucional dos metadados da série.
- Federal Reserve Bank of St. Louis (FRED): `DGS10` e `DFF`.
- Seabold e Perktold, *statsmodels*; documentação de SARIMAX, Exponential Smoothing, STL e Ljung-Box.
- Pedregosa et al., *Scikit-learn: Machine Learning in Python*; documentação de Random Forest, `StandardScaler`, permutation importance e `PLSRegression`.
- Wold (1975), *Path Models with Latent Variables: The NIPALS Approach*, referência conceitual de PLS.

Referências e sínteses globais pertencem ao relatório consolidado posterior.

In [ ]:
display(table('pls_feature_interpretation.csv'))
figure('pls_feature_interpretation.png')